# Band Structure Calculation Demo

This notebook demonstrates calculating photonic band structures and band gaps using DFT-inspired methods.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
from src.data.photonic_crystal_generator import PhotonicCrystalGenerator
from src.dft.band_structure_calculator import BandStructureCalculator
from src.dft.transmission_calculator import TransmissionCalculator
from src.utils.visualization import plot_band_structure, plot_transmission, plot_dos

## Create Photonic Crystal

In [ ]:
generator = PhotonicCrystalGenerator(random_seed=42)

# Create a 2D square lattice
crystal = generator.generate_2d_square_lattice(
    n_background=1.0,
    n_rod=3.4,
    lattice_constant=0.5,
    rod_radius=0.3,
    grid_size=64
)

print(f"Crystal: {crystal.lattice_type}")
print(f"Lattice constant: {crystal.lattice_constant} μm")
print(f"Refractive index contrast: {max(crystal.refractive_indices) / min(crystal.refractive_indices):.2f}")

## Calculate Band Structure

In [ ]:
# Initialize band structure calculator
band_calc = BandStructureCalculator(
    num_g_vectors=289,  # 17x17 reciprocal lattice vectors
    num_k_points=100
)

# Calculate band structure
print("Calculating band structure...")
k_points, frequencies = band_calc.calculate_band_structure(crystal)

print(f"k-points shape: {k_points.shape}")
print(f"Frequencies shape: {frequencies.shape}")
print(f"Number of bands: {frequencies.shape[1]}")

## Identify Band Gap

In [ ]:
# Calculate band gap
band_gap_info = band_calc.calculate_band_gap(crystal)

print("Band Gap Information:")
print(f"  Band gap: {band_gap_info['band_gap']:.6f} (ωa/2πc)")
print(f"  Gap ratio: {band_gap_info['gap_ratio']:.4f}")
print(f"  Lower band edge: {band_gap_info['lower_band_edge']:.6f}")
print(f"  Upper band edge: {band_gap_info['upper_band_edge']:.6f}")
print(f"  Band index: {band_gap_info['band_index']}")

## Visualize Band Structure

In [ ]:
fig = plot_band_structure(
    k_points,
    frequencies,
    band_gap_info=band_gap_info,
    title=f"Photonic Band Structure - {crystal.lattice_type}"
)
plt.show()

## Calculate Density of States

In [ ]:
# Calculate DOS
print("Calculating density of states...")
freq_bins, dos = band_calc.calculate_dos(crystal, num_k_samples=50)

fig = plot_dos(
    freq_bins,
    dos,
    band_gap_info=band_gap_info,
    title="Photonic Density of States"
)
plt.show()

## Calculate Transmission Spectrum

In [ ]:
# Create a 1D multilayer for transmission calculation
crystal_1d = generator.generate_1d_multilayer(
    n1=1.5,
    n2=3.5,
    num_layers=10,
    thickness_ratio=0.5,
    lattice_constant=0.5
)

# Initialize transmission calculator
trans_calc = TransmissionCalculator(
    wavelength_range=(0.3, 2.0),
    num_wavelengths=200
)

print("Calculating transmission spectrum...")
result = trans_calc.calculate_transmission_spectrum(crystal_1d)

wavelengths = result['wavelengths']
transmission = result['transmission']
reflection = result['reflection']

In [ ]:
fig = plot_transmission(
    wavelengths,
    transmission,
    reflection=reflection,
    title="Transmission Spectrum - 1D Multilayer"
)
plt.show()

## Compare Different Crystal Structures

In [ ]:
# Generate different crystals and calculate their band gaps
crystals_to_compare = [
    generator.generate_2d_square_lattice(1.0, 3.4, 0.5, 0.2, 32),
    generator.generate_2d_square_lattice(1.0, 3.4, 0.5, 0.3, 32),
    generator.generate_2d_square_lattice(1.0, 3.4, 0.5, 0.4, 32),
]

rod_radii = [0.2, 0.3, 0.4]
band_gaps = []

for crystal in crystals_to_compare:
    bg_info = band_calc.calculate_band_gap(crystal)
    band_gaps.append(bg_info['band_gap'])

# Plot band gap vs rod radius
plt.figure(figsize=(10, 6))
plt.plot(rod_radii, band_gaps, 'o-', linewidth=2, markersize=10)
plt.xlabel('Rod Radius (relative to lattice constant)', fontsize=12)
plt.ylabel('Band Gap (ωa/2πc)', fontsize=12)
plt.title('Band Gap vs Rod Radius', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

print("\nBand Gap Analysis:")
for r, bg in zip(rod_radii, band_gaps):
    print(f"  Rod radius {r:.1f}: Band gap = {bg:.6f}")

## Quality Factor Calculation

In [ ]:
# Calculate Q-factor for 1D multilayer
q_factor = trans_calc.calculate_quality_factor(crystal_1d)
print(f"Quality factor (Q): {q_factor:.2f}")

## Summary

This notebook demonstrated:
- Calculating photonic band structures using plane wave expansion
- Identifying and visualizing band gaps
- Computing density of states
- Calculating transmission spectra
- Analyzing effects of structural parameters

Next: See `03_gnn_training.ipynb` for training GNN models on this data.